## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and connects the data from Google Drive.

**Before you run it**, make sure you've opened the shared camp Drive folder and
clicked **"Add shortcut to Drive"** (put the shortcut in *My Drive*) — that's how
the notebook finds the data file. Then run the cell below and click **Connect** on
the Drive pop-up. Wait for **✅ Setup complete**, then run the rest top to bottom.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os, sys, glob

print("1/3  installing mne ...")
get_ipython().system('pip install -q "mne==1.10.1"')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  connecting Google Drive for the data ...")
from google.colab import drive
drive.mount("/content/drive")
hits = sorted(glob.glob("/content/drive/MyDrive/**/synapse_preprocessed.pkl", recursive=True))
assert hits, (
    "Could not find synapse_preprocessed.pkl in your Drive.\n"
    "Open the shared camp folder, click 'Add shortcut to Drive', put the shortcut "
    "in 'My Drive', then run this cell again."
)
os.environ["CAMP_DATA_PATH"] = hits[0]
os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
print(f"\n\u2705 Setup complete. Using data at: {hits[0]}")
print("Your figures will be saved to Drive > DecodingBrain_outputs.")


# Week 3 · Tier 1 — Spatial Maps (Where on the ear?)

**Tier 1 "EEG Explorer"** focuses on beautiful, clear visualizations. This
notebook answers **Research Goal 5 (the "where" part)**:

> Which of the 16 ear electrodes show the biggest group differences?

Each ear has electrodes in three regions — **Tragus** (front of the ear),
**Mastoid** (bone behind the ear), and **Temporal** (up toward the temple).
Does location matter?

### By the end of this notebook you will be able to
1. Compute a feature **per channel** (not averaged across the ear)
2. Build a channel × group comparison
3. Map each electrode to its anatomical region
4. Make a clean "which electrode lights up" figure for your poster

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import camp_utils as cu

data = cu.load_camp_data(verbose=False)

## 1. Per-channel band power
Until now we averaged across all electrodes to get one number. To make a *map*,
we keep each channel separate. `cu.band_power_db_per_channel` returns one dB
value per channel.

In [ ]:
ep = data["exp_epochs"]["let"][0]
window = cu.get_time_windows("let")["full_stim"]
per_ch = cu.band_power_db_per_channel(ep, "gamma", window)

for ch, val in zip(ep.ch_names, per_ch):
    print(f"  {ch}: {val:+.2f} dB")

## 2. Average each channel across a whole group
Channels can be missing for some subjects, so we average channel-by-channel using
its name as the key. Read this helper, then run it.

In [ ]:
def group_channel_means(data, group, task, band, period="full_stim"):
    """Return {channel_name: mean dB across the group} for one band."""
    window = cu.get_time_windows(task)[period]
    collected = {}   # channel -> list of values
    for subject, ep in cu.iter_subjects(data, group, task):
        vals = cu.band_power_db_per_channel(ep, band, window)
        if vals is None:
            continue
        for ch, v in zip(ep.ch_names, vals):
            if not np.isnan(v):
                collected.setdefault(ch, []).append(v)
    return {ch: np.mean(v) for ch, v in collected.items()}

exp_ch = group_channel_means(data, "exp", "let", "gamma")
ctrl_ch = group_channel_means(data, "ctrl", "let", "gamma")
print("EXP channels measured:", len(exp_ch))

## 3. Build a per-channel comparison table
### ✏️ Your turn #1 — fill in the difference
For each channel present in both groups, compute EXP mean − CTRL mean.

In [ ]:
all_channels = sorted(set(exp_ch) & set(ctrl_ch))   # channels both groups have

rows = []
for ch in all_channels:
    # TODO: compute the difference for this channel
    diff = None   # exp_ch[ch] - ctrl_ch[ch]
    rows.append({
        "channel": ch,
        "region": cu.get_electrode_region(ch),
        "ear": "Left" if ch.startswith("L") else "Right",
        "exp": exp_ch[ch],
        "ctrl": ctrl_ch[ch],
        "diff": diff,
    })

chan = pd.DataFrame(rows)
cu.check(chan["diff"].notna().all(),
         "Per-channel difference table is filled.",
         "diff should be exp_ch[ch] - ctrl_ch[ch].")
print(chan.round(2).to_string(index=False))

## 4. Which channel differs most?

In [ ]:
top = chan.loc[chan["diff"].abs().idxmax()]
print(f"Biggest group difference: {top['channel']} "
      f"({top['region']}, {top['ear']} ear) = {top['diff']:+.2f} dB")

## 5. A spatial bar map
A simple, honest "map": one bar per channel, colored by region, sorted by ear.
Tall bars = electrodes where the groups differ most.

In [ ]:
chan_sorted = chan.sort_values(["ear", "channel"]).reset_index(drop=True)
colors = [cu.REGION_COLORS[r] for r in chan_sorted["region"]]

fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.bar(chan_sorted["channel"], chan_sorted["diff"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("EXP − CTRL gamma (dB)")
ax.set_title("Group difference at each ear electrode (LET, gamma)")
# a legend for the regions
from matplotlib.patches import Patch
legend = [Patch(facecolor=c, label=r) for r, c in cu.REGION_COLORS.items()]
ax.legend(handles=legend, title="Region")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(cu.save_path("tier1_spatial_map.png"), dpi=300, bbox_inches="tight")
plt.show()

## 6. Does region matter? (group by anatomy)

In [ ]:
region_summary = chan.groupby("region")["diff"].agg(["mean", "count"])
print(region_summary.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
region_means = chan.groupby("region")["diff"].mean()
ax.bar(region_means.index, region_means.values,
       color=[cu.REGION_COLORS[r] for r in region_means.index])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Mean EXP − CTRL gamma (dB)")
ax.set_title("Average group difference by ear region")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### ✏️ Your turn #2 — your own spatial map
Pick a **different band and/or task** (e.g. alpha during HLT) and rebuild the
whole per-channel map. Save it to `outputs/`. Then answer (as a comment):
- Which region shows the strongest difference for your choice?
- Is it the same region as the gamma/LET map above, or different?

In [ ]:
# TODO: rebuild group_channel_means(...) for your band+task, make the table,
#       and draw the spatial bar map. Save with cu.save_path(...).

## 🎯 Wrap-up (Tier 1)
You turned a single number into a **map** of the ear, colored by anatomy. This is
exactly the kind of clear, accessible figure that makes a poster shine.

**For your poster:** a spatial map + one sentence ("group differences were
strongest at the X electrodes, in the Y region") is a complete, compelling result.